# UNet — Extraction de lignes de côte de falaises par segmentation sémantique
## Version 12 — Pipeline complet avec validation de loss et post-traitement

Ce notebook implémente un réseau de neurones UNet pour la détection automatique
de la ligne de côte de falaises à partir d'images satellites Pléiades (résolution 0.5m).
La tâche est une segmentation binaire pixel-à-pixel : chaque pixel est classifié
comme appartenant à la côte (1) ou au fond (0).

### Contexte et enjeux
La ligne de côte représente moins de 1% des pixels de l'image, ce qui crée un
fort déséquilibre de classes. Les choix méthodologiques (loss combinée BCE+Dice,
pos_weight, filtrage des tuiles vides, stratification IPS) visent tous à compenser
ce déséquilibre.

### Structure du notebook
| Section | Contenu |
|---|---|
| 1 | Imports et chemins |
| 2 | Dataset et augmentation |
| 3 | Filtrage + Stratification IPS + DataLoaders |
| 4 | Architecture UNet |
| 5 | Loss combinée BCE + Dice |
| 6 | Métriques d'évaluation |
| 7 | Diagnostic pré-entraînement |
| 8 | Boucle d'entraînement |
| 9 | Courbes ROC et Precision-Recall |
| 10 | Courbe de loss (train + validation) |
| 11 | Évaluation finale |
| 12 | Inférence sur nouvelles images |
| Annexe | Visualisation qualitative des prédictions |

### Paramètres clés à ajuster avant lancement
- `MIN_COAST_PIXELS` (section 3) : seuil de filtrage des tuiles vides
- `num_epochs` (section 8) : nombre d'époques d'entraînement
- `pos_weight_value` (section 3) : pondération de la classe minoritaire (côte)
  — valeur optimale déterminée empiriquement : **10**


## 1 — Imports et chemins

Cette section charge toutes les bibliothèques nécessaires et définit les chemins
vers les données. Chaque import est justifié par son rôle dans le pipeline.


In [ ]:

# --- Bibliothèques standard Python ---
import os, random          # gestion des chemins et reproductibilité du mélange aléatoire
import numpy as np         # calculs matriciels (manipulation des tableaux d'images)

# --- PyTorch : framework de deep learning ---
import torch                          # tenseurs et calcul GPU/CPU
import torch.nn as nn                 # couches de réseaux de neurones (Conv2d, BatchNorm, etc.)
import torch.nn.functional as F       # fonctions sans paramètres (sigmoid, interpolate, etc.)
from torch.utils.data import Dataset, DataLoader, Subset
# Dataset   : classe abstraite pour définir un jeu de données personnalisé
# DataLoader: itérateur qui charge les données par batch pendant l'entraînement
# Subset    : sous-ensemble d'un Dataset sélectionné par indices (utilisé pour le split train/val/test)

# --- Albumentations : bibliothèque d'augmentation synchronisée image/masque ---
# Contrairement à torchvision.transforms, albumentations applique exactement
# la même transformation géométrique à l'image ET au masque de segmentation,
# ce qui est indispensable pour maintenir la cohérence des paires (image, masque).
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- Visualisation ---
import matplotlib.pyplot as plt       # tracé des courbes de loss, ROC, et visualisation des prédictions
from tqdm import tqdm                 # barre de progression pendant l'entraînement
from glob import glob                 # listing de fichiers par pattern (ex: '*.tif')
from PIL import Image                 # chargement et conversion d'images
from scipy import ndimage             # Filtrage lignes
from scipy.ndimage import binary_dilation #Dilatation pour visualisation

# --- Rasterio : lecture/écriture de fichiers géographiques (.tif, .jp2) ---
import rasterio                       # lecture des images satellites géoréférencées
from rasterio.windows import Window   # définit une fenêtre de découpe (tuile) dans une grande image
from rasterio.merge import merge      # fusionne plusieurs tuiles en une seule mosaïque
from rasterio.transform import Affine # calcule la géoréférence de chaque tuile découpée
from rasterio.crs import CRS          # système de référence de coordonnées (ex: EPSG:2154 Lambert 93)

# --- Métriques scikit-learn ---
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve
# roc_curve : calcule les taux vrais/faux positifs à chaque seuil possible
# auc       : calcule l'aire sous la courbe ROC (AUC), indicateur global de discrimination

# --- Bibliothèque de stratification sémantique ---
import sys
sys.path.append(r'C:\Users\Rhm\SemanticStratification')
from stratifiers.ips import IPSKFold

# --- Post-traitement ---
from scipy import ndimage

# --- Répertoire de travail et chemins des données ---
os.chdir(r'C:\Users\Rhm\Documents\M2 SIC\Projet de recherche\dataset\test 3')

data_root = r'C:\Users\Rhm\Documents\M2 SIC\Projet de recherche\dataset\test 3\dataset_stride'
image_dir = os.path.join(data_root, 'Images')  # dossier contenant les tuiles images (RGB)
mask_dir  = os.path.join(data_root, 'Masks')   # dossier contenant les masques binaires associés

# Vérification que les dossiers existent avant de continuer
assert os.path.exists(image_dir), f'Introuvable : {image_dir}'
assert os.path.exists(mask_dir),  f'Introuvable : {mask_dir}'
print('Chemins OK')

## 2 — Dataset et augmentation

### Normalisation ImageNet
Les images sont normalisées avec les statistiques d'ImageNet (moyenne et écart-type
par canal R, G, B). Cette normalisation est standard même sans modèle pré-entraîné
car elle place les valeurs d'entrée dans une plage favorable à l'optimisation.
Elle est appliquée **identiquement** sur train, val, test et lors de l'inférence.

### Augmentation de données
L'augmentation est appliquée **uniquement au train set** pour éviter un biais
lors de l'évaluation. La bibliothèque `albumentations` garantit que la même
transformation géométrique est appliquée simultanément à l'image ET à son masque,
ce qui est indispensable en segmentation sémantique.

Deux transformations ont été retenues après ablation empirique :

| Transformation | Justification domaine |
|---|---|
| `HorizontalFlip` | Une côte vue de gauche ou de droite est sémantiquement identique — augmentation sans biais |
| `ColorJitter` | Simule les variations d'acquisition : heure de prise de vue, saison, différences entre capteurs |

Les autres transformations (Rotate, RandomResizedCrop, GaussianBlur, GaussNoise)
ont été désactivées car elles dégradaient les performances sur ce petit dataset
(318 tuiles d'entraînement) — le modèle ne dispose pas de suffisamment d'exemples
pour apprendre à être robuste à autant de variabilité simultanée.


In [ ]:
# Valeurs de normalisation ImageNet (moyenne et écart-type par canal R, G, B)
# Ces valeurs sont issues de l'entraînement d'ImageNet et sont standard
# pour les modèles pré-entraînés. On les applique de manière cohérente
# sur tous les splits (train, val, test) et lors de l'inférence.
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# Taille des tuiles 
TILE_SIZE = 256


def get_train_transform():
    """Pipeline d'augmentation pour le jeu d'entraînement.

    albumentations applique exactement la même transformation géométrique
    à l'image et au masque (via additional_targets), ce qui est indispensable
    en segmentation sémantique pour maintenir la cohérence des paires.
    """
    return A.Compose([
        # --- Augmentations géométriques ---
        # HorizontalFlip : une côte vue de gauche ou de droite est identique
        # sémantiquement → augmentation sans biais pour ce domaine.
        A.HorizontalFlip(p=0.5),

        # ±15° couvre les variations d'orientation réalistes pour des acquisitions aériennes.
        # border_mode=0 : remplissage noir aux bords (évite la répétition des pixels de bord).
        ##A.Rotate(
        #    limit=15,
        #    border_mode=0,
        #    p=0.6,
        ##),

        # RandomResizedCrop : simule les variations d'altitude de vol et de zoom capteur.
        # scale=(0.70, 1.0) : le modèle voit parfois seulement 70% de la tuile zoomée
        # à 256×256, ce qui l'entraîne à reconnaître la falaise à différentes résolutions.
        # ratio=(0.9, 1.1) : proche carré, cohérent avec les tuiles carrées du dataset.
        ##A.RandomResizedCrop(
        #    size=(TILE_SIZE, TILE_SIZE), 
        #    scale=(0.70, 1.0), 
        #    ratio=(0.9, 1.1), 
        #    p=0.5
        ##),

        # --- Augmentations colorimétriques ---
        # ColorJitter : simule les conditions d'acquisition variables
        # (heure de la journée, couverture nuageuse, différences entre capteurs).
        # saturation réduite (0.05) : mer et roche ont des palettes naturellement
        # peu saturées, une perturbation forte serait non réaliste.
        A.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.05,
            hue=0.02,
            p=0.7,
        ),

        # GaussianBlur : simule le flou de bougé (mouvement de la plateforme)
        # et les distorsions atmosphériques lors des acquisitions aériennes.
        # blur_limit=(3,5) : noyaux 3×3 et 5×5, valeurs réalistes pour ce type d'images.
        ##A.GaussianBlur(
        #    blur_limit=(3, 5),
        #    sigma_limit=(0, 1),
        #    p=0.3,
        ##),

        # GaussNoise : simule le bruit électronique du capteur.
        # var_limit=(10, 30) : amplitude faible pour rester dans le domaine réaliste.
        ##A.GaussNoise(
        #    var_limit=(10.0, 30.0),
        #    p=0.25,
        ##),

        # --- Normalisation et conversion tenseur ---
        # Normalize : (pixel - mean) / std par canal, indispensable pour que
        # les valeurs d'entrée soient dans une plage compatible avec les poids du réseau.
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),

        # ToTensorV2 : convertit [H,W,3] numpy uint8 → tenseur PyTorch [3,H,W] float32.
        # Contrairement à ToTensor de torchvision, ToTensorV2 ne divise PAS par 255
        # (c'est Normalize qui s'en charge) et ne traite pas le masque comme une image.
        ToTensorV2(),
    ],
    # additional_targets garantit que le masque reçoit exactement
    # la même transformation géométrique que l'image.
    additional_targets={'mask': 'mask'},
    )


def get_val_transform():
    """Pipeline de transformations pour val, test et inférence.
    Pas d'augmentation : on veut une évaluation reproductible et stable.
    """
    return A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ],
    additional_targets={'mask': 'mask'},
    )


class CoastlineDataset(Dataset):
    """Classe Dataset personnalisée pour charger les paires (image, masque).
    Hérite de torch.utils.data.Dataset et implémente __len__ et __getitem__
    comme requis par PyTorch.
    """
    def __init__(self, image_dir, mask_dir, img_transform=None):
        # Liste triée de tous les fichiers images et masques.
        # Le tri garantit que image[i] et mask[i] correspondent toujours.
        self.image_paths = sorted(glob(os.path.join(image_dir, '*.*')))
        self.mask_paths  = sorted(glob(os.path.join(mask_dir,  '*.*')))
        self.transform   = img_transform
        # Vérification de cohérence : autant d'images que de masques
        assert len(self.image_paths) == len(self.mask_paths), \
            'Nombre images / masques différent'

    def __len__(self):
        # Retourne le nombre total de tuiles dans le dataset
        return len(self.image_paths)

    def __getitem__(self, idx):
        """Charge et retourne la paire (image, masque) à l'indice idx."""

        # Lecture de l'image satellite via rasterio.
        # src.read([1,2,3]) lit les 3 bandes (R,G,B) → format [3,H,W].
        # np.moveaxis(..., 0, -1) convertit en [H,W,3] attendu par albumentations.
        with rasterio.open(self.image_paths[idx]) as src:
            img_np = np.moveaxis(src.read([1, 2, 3]), 0, -1).astype(np.uint8)

        # Lecture du masque binaire (bande unique).
        # Seuil à 0.0001 pour convertir en masque binaire {0, 1} :
        # tout pixel > 0.0001 est considéré comme appartenant à la côte.
        with rasterio.open(self.mask_paths[idx]) as src:
            mask_raw = src.read(1).astype(np.float32)
        mask_bin = (mask_raw > 0.0001).astype(np.uint8)  # {0, 1} uint8 pour albumentations

        if self.transform:
            # albumentations attend image en [H,W,3] uint8 et mask en [H,W] uint8
            augmented = self.transform(image=img_np, mask=mask_bin)
            img_tensor  = augmented['image']            # tenseur [3,H,W] float32
            mask_tensor = augmented['mask'].float()     # tenseur [H,W] float32 {0., 1.}
        else:
            # Fallback sans transform : conversion manuelle en tenseur
            img_tensor  = torch.from_numpy(img_np.transpose(2, 0, 1)).float() / 255.0
            mask_tensor = torch.from_numpy(mask_bin).float()

        return img_tensor, mask_tensor


print('Dataset et transforms albumentations établis')

## 3 — Filtrage des tuiles vides + Stratification IPS + DataLoaders

### Filtrage des tuiles
Le dataset de 2310 tuiles contient une majorité de tuiles sans ligne de côte
(mer ou terre seules). Ces tuiles n'apportent aucune information utile à
l'apprentissage de la côte et introduisent du bruit. On ne conserve que les
tuiles contenant au moins `MIN_COAST_PIXELS` pixels de côte.

| Valeur `MIN_COAST_PIXELS` | Effet |
|---|---|
| 5–10 | Conserve presque tout, élimine seulement les tuiles vraiment vides |
| 20–50 | Filtrage modéré — **valeur retenue : 50** |
| > 100 | Filtrage agressif, risque de perdre des tuiles utiles |

### Stratification IPS (Iterative Proportional Stratification)
La stratification garantit que la **proportion de pixels de côte est équilibrée**
entre les splits train, val et test. Sans stratification, le tirage aléatoire
pourrait concentrer les tuiles riches en côte dans un seul split et biaiser
l'évaluation.

Découpage : 10 folds → **80% train / 10% val / 10% test** (folds 2-9 / fold 1 / fold 0).

### pos_weight
Le paramètre `pos_weight` de `BCEWithLogitsLoss` pondère la classe minoritaire
(côte). Avec ~0.65% de pixels côte, le ratio théorique est ~150, mais la valeur
empiriquement optimale est **10** — la Dice Loss compensant déjà une partie du
déséquilibre, un pos_weight trop élevé sur-corrigerait et dégraderait la précision.


In [ ]:
# ═══════════════════════════════════════════════
# PARAMÈTRE À AJUSTER
MIN_COAST_PIXELS = 50
# ═══════════════════════════════════════════════

def filter_dataset(dataset, min_coast_pixels):
    """Retourne les indices des tuiles contenant au moins min_coast_pixels pixels de côte."""
    print(f'Filtrage (seuil = {min_coast_pixels} px)')
    kept, empty, total_coast = [], 0, 0
    for i in range(len(dataset)):
        _, mask = dataset[i]
        n_coast = mask.sum().item()
        if n_coast >= min_coast_pixels:
            kept.append(i)
            total_coast += n_coast
        else:
            empty += 1
    print(f'  Tuiles conservées : {len(kept):4d} / {len(dataset)} '
          f'({100*len(kept)/len(dataset):.1f}%)')
    print(f'  Tuiles vides exclues : {empty}')
    print(f'  Pixels côte totaux : {total_coast:,}')
    print(f'  Moyenne px côte / tuile : {total_coast/max(len(kept),1):.1f}')
    return kept


def compute_pos_weight(dataset, max_samples=200):
    """Ratio négatifs/positifs → pos_weight pour BCEWithLogitsLoss."""
    pos, neg = 0, 0
    for i in range(min(len(dataset), max_samples)):
        _, mask = dataset[i]
        p   = mask.sum().item()
        pos += p
        neg += mask.numel() - p
    ratio   = neg / (pos + 1e-8)
    clamped = min(ratio, 300.0)
    print(f'\nRatio négatifs/positifs : {ratio:.1f}')
    print(f'pos_weight utilisé       : {clamped:.1f} '
          f'({"plafonné" if ratio > 300 else "valeur réelle"})')
    return clamped


class StratificationWrapper:
    """Adapte CoastlineDataset au format attendu par IPSKFold.
    IPSKFold attend :
      - dataset.num_classes : nombre de classes (2 : fond + côte)
      - itération (_, mask) où mask est un np.array d'entiers {0, 1}
    """
    def __init__(self, indices, base_dataset):
        self.indices     = indices
        self.base        = base_dataset
        self.num_classes = 2

    def __len__(self):
        return len(self.indices)

    def __iter__(self):
        for i in self.indices:
            _, mask = self.base[i]
            yield None, mask.numpy().astype(np.int64)


# --- Filtrage ---
# On utilise get_val_transform() pour le filtrage car on veut évaluer
# le contenu réel des masques sans augmentation aléatoire.
base_ds  = CoastlineDataset(image_dir, mask_dir, img_transform=get_val_transform())
kept_idx = filter_dataset(base_ds, MIN_COAST_PIXELS)

# --- Stratification IPS ---
# La stratification garantit une distribution équilibrée des pixels de côte
# entre les splits train, val et test.
strat_ds = StratificationWrapper(kept_idx, base_ds)

# n_splits=10 → fold 0=test (10%), fold 1=val (10%), folds 2-9=train (80%)
ips      = IPSKFold(n_splits=10, shuffle=False)
all_folds = list(ips.split(strat_ds))

_, test_local  = all_folds[0]
_, val_local   = all_folds[1]
train_local    = np.concatenate([all_folds[i][1] for i in range(2, 10)])

train_idx = [kept_idx[i] for i in train_local]
val_idx   = [kept_idx[i] for i in val_local]
test_idx  = [kept_idx[i] for i in test_local]

print(f'\nSplit stratifié IPS : Train={len(train_idx)} | Val={len(val_idx)} | Test={len(test_idx)}')

# Vérification de l'équilibre en pixels de côte entre les splits
for name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    coast_px = sum(base_ds[i][1].sum().item() for i in idx)
    total_px = sum(base_ds[i][1].numel() for i in idx)
    print(f'  {name:5} : {len(idx):3d} tuiles | '
          f'{coast_px:,.0f} px côte ({100*coast_px/total_px:.2f}%)')

# --- Datasets et dataloaders ---
# Le train dataset utilise get_train_transform() (avec augmentation albumentations).
# Les datasets de val et test utilisent get_val_transform() (sans augmentation).
full_train = CoastlineDataset(image_dir, mask_dir, img_transform=get_train_transform())
full_val   = CoastlineDataset(image_dir, mask_dir, img_transform=get_val_transform())

train_dataset = Subset(full_train, train_idx)
val_dataset   = Subset(full_val,   val_idx)
test_dataset  = Subset(full_val,   test_idx)

train_dl = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=0, pin_memory=False)
val_dl   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=0, pin_memory=False)
test_dl  = DataLoader(test_dataset,  batch_size=4, shuffle=False, num_workers=0, pin_memory=False)

pos_weight_value = 10

## 4 — Architecture UNet

Le UNet (Ronneberger et al., 2015) est une architecture encodeur-décodeur
conçue pour la segmentation sémantique. Son originalité réside dans les
**skip connections** qui relient directement chaque niveau de l'encodeur
au niveau symétrique du décodeur.

```
Image 256×256
     │
  Encoder1 ──────────────────────────────────► Decoder1
     │ pool                                        ▲ cat
  Encoder2 ────────────────────────────► Decoder2  │
     │ pool                                  ▲ cat │
  Encoder3 ──────────────────────► Decoder3  │     │
     │ pool                           ▲ cat  │     │
  Encoder4 ──────────────► Decoder4   │      │     │
     │ pool                    ▲ cat  │      │     │
  Bottleneck                   │      │      │     │
```

**Pourquoi les skip connections sont essentielles ici :**
La ligne de côte est une structure fine (1-2 pixels). Sans skip connections,
l'information spatiale précise serait perdue lors de la compression par MaxPooling.
Les skip connections permettent au décodeur de récupérer cette information
spatiale fine depuis l'encodeur.

`forward()` retourne des **logits bruts** (avant sigmoid). La sigmoid est
appliquée dans la loss pour la stabilité numérique, et explicitement
lors de l'inférence.


In [ ]:
# Sélection automatique du GPU si disponible, sinon CPU.
# Le GPU accélère considérablement l'entraînement (×10 à ×50 selon le matériel).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')


class UNet(nn.Module):
    """Architecture UNet pour la segmentation sémantique pixel-à-pixel.

    Le UNet est composé de deux chemins symétriques :
    - Encodeur (chemin descendant) : extrait des features de plus en plus abstraites
      en réduisant la résolution spatiale par MaxPooling
    - Décodeur (chemin ascendant) : reconstruit la carte de segmentation à la
      résolution originale par upsampling bilinéaire
    Les skip connections (torch.cat) entre encodeur et décodeur permettent de
    combiner les features locales (détails) et globales (contexte), ce qui est
    essentiel pour détecter des structures fines comme une ligne de côte.
    """
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        # --- Encodeur : 4 niveaux de résolution décroissante ---
        # Chaque bloc double le nombre de filtres (64→128→256→512).
        # Plus de filtres = features plus abstraites et discriminantes.
        self.encoder1   = self._block(in_channels, 64)
        self.encoder2   = self._block(64,  128)
        self.encoder3   = self._block(128, 256)
        self.encoder4   = self._block(256, 512)
        # --- Goulot d'étranglement (bottleneck) ---
        # Représentation la plus compressée et abstraite de l'image.
        self.bottleneck = self._block(512, 1024)
        # --- Décodeur : reconstruction symétrique ---
        # +512, +256, +128, +64 : canaux des skip connections concaténées.
        self.decoder4   = self._block(1024+512, 512)
        self.decoder3   = self._block(512 +256, 256)
        self.decoder2   = self._block(256 +128, 128)
        self.decoder1   = self._block(128 + 64,  64)
        # Couche finale : 64 canaux → 1 canal (probabilité de côte par pixel).
        # kernel_size=1 : convolution 1×1, équivalent à une classification par pixel.
        self.final      = nn.Conv2d(64, out_channels, kernel_size=1)
        # MaxPool2d(2) : divise la résolution spatiale par 2 à chaque niveau.
        self.pool       = nn.MaxPool2d(2)

    def _block(self, in_ch, out_ch):
        """Bloc convolutif de base : Conv → BN → ReLU → Conv → BN → ReLU.
        - Conv2d(3, padding=1) : convolution 3×3 qui préserve la résolution spatiale
        - BatchNorm2d : normalise les activations pour stabiliser l'entraînement
        - ReLU(inplace=True) : activation non-linéaire (inplace économise la mémoire)
        """
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def _up(self, x):
        """Upsampling bilinéaire : double la résolution spatiale.
        Préféré à la transposée convolutive car plus stable et sans artefacts
        de quadrillage (checkerboard artifacts).
        """
        return F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)

    def forward(self, x):
        """Propagation avant : calcule la prédiction pour un batch d'images.
        Retourne des LOGITS bruts (avant sigmoid) — la fonction sigmoid est
        appliquée dans la loss (BCEWithLogitsLoss) pour la stabilité numérique,
        et explicitement lors de l'évaluation/inférence.
        """
        # Chemin encodeur : extraction de features + réduction résolution
        e1 = self.encoder1(x)               # [B, 64,  H,   W  ]
        e2 = self.encoder2(self.pool(e1))   # [B, 128, H/2, W/2]
        e3 = self.encoder3(self.pool(e2))   # [B, 256, H/4, W/4]
        e4 = self.encoder4(self.pool(e3))   # [B, 512, H/8, W/8]
        b  = self.bottleneck(self.pool(e4)) # [B,1024, H/16,W/16]
        # Chemin décodeur : upsampling + skip connections (torch.cat).
        # torch.cat concatène les features upsamplées avec celles de l'encodeur
        # correspondant → le modèle combine information locale et globale.
        d4 = self.decoder4(torch.cat([self._up(b),  e4], dim=1))  # [B,512, H/8, W/8]
        d3 = self.decoder3(torch.cat([self._up(d4), e3], dim=1))  # [B,256, H/4, W/4]
        d2 = self.decoder2(torch.cat([self._up(d3), e2], dim=1))  # [B,128, H/2, W/2]
        d1 = self.decoder1(torch.cat([self._up(d2), e1], dim=1))  # [B,64,  H,   W  ]
        return self.final(d1)   # [B, 1, H, W] — logits bruts


model = UNet().to(device)
print(f'Paramètres : {sum(p.numel() for p in model.parameters()):,}')

## 5 — Loss combinée BCE + Dice

Une loss unique serait insuffisante pour ce problème :
- La **BCE seule** traite chaque pixel indépendamment et est dominée par les
  pixels de fond (majoritaires à ~99.35%).
- La **Dice Loss seule** optimise le chevauchement global mais n'offre pas
  de correction directe du déséquilibre de classes.

La **combinaison 50/50** BCE pondérée + Dice Loss bénéficie des avantages
des deux approches de façon complémentaire :

| Composante | Rôle | Paramètre clé |
|---|---|---|
| BCE pondérée | Corrige le déséquilibre à l'échelle du pixel | `pos_weight=10` |
| Dice Loss | Optimise le chevauchement, adaptée aux structures fines | `smooth=1.0` |


In [ ]:
def dice_loss(logits, targets, smooth=1.0):
    """Dice Loss : mesure le chevauchement entre prédiction et vérité terrain.

    Formule : 1 - (2 * |P ∩ T| + smooth) / (|P| + |T| + smooth)
    Le paramètre smooth évite la division par zéro quand une tuile
    ne contient aucun pixel de côte.

    La Dice Loss est particulièrement adaptée aux structures fines car elle
    optimise directement le ratio de recouvrement, indépendamment du nombre
    total de pixels — contrairement à la BCE qui traite chaque pixel séparément.
    """
    probs  = torch.sigmoid(logits)  # conversion logits → probabilités [0,1]
    p_flat = probs.view(-1)          # mise à plat en vecteur 1D
    t_flat = targets.view(-1)
    inter  = (p_flat * t_flat).sum() # intersection pondérée par les probabilités
    return 1 - (2. * inter + smooth) / (p_flat.sum() + t_flat.sum() + smooth)


def combined_loss(logits, targets, pos_weight, w_bce=0.5, w_dice=0.5):
    """Loss combinée : BCE pondérée (50%) + Dice Loss (50%).

    - BCE pondérée (BCEWithLogitsLoss) : pénalise davantage les erreurs sur
      les pixels de côte (minoritaires) via pos_weight. Corrige le déséquilibre
      global de classes à l'échelle du pixel.
    - Dice Loss : optimise directement le chevauchement prédit/GT. Complémentaire
      de la BCE car elle se concentre sur la qualité de détection des structures
      fines sans être influencée par le nombre de pixels de fond.
    La combinaison 50/50 offre un bon équilibre entre les deux objectifs.
    """
    bce_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    l_bce  = bce_fn(logits.squeeze(1), targets)   # squeeze : [B,1,H,W] → [B,H,W]
    l_dice = dice_loss(logits.squeeze(1), targets)
    return w_bce * l_bce + w_dice * l_dice


print('Loss combinée BCE + Dice prête.')

## 6 — Métriques d'évaluation

Les métriques sont calculées après binarisation des probabilités au seuil choisi.
L'**accuracy** est volontairement incluse mais doit être interprétée avec précaution :
un modèle prédisant tout 'fond' obtiendrait ~99.4% d'accuracy sans détecter
un seul pixel de côte. Les métriques pertinentes sont le **F1-score** et l'**IoU côte**.

| Métrique | Formule | Interprétation |
|---|---|---|
| IoU côte | TP / (TP+FP+FN) | Chevauchement prédit/réel — métrique principale |
| mIoU | moyenne(IoU_côte, IoU_fond) | Standard littérature segmentation |
| F1 | 2·P·R / (P+R) | Équilibre précision/recall — robuste au déséquilibre |
| Précision | TP / (TP+FP) | Fraction de pixels côte prédits corrects |
| Recall | TP / (TP+FN) | Fraction de la côte réelle détectée |
| Accuracy | (TP+TN) / total | Trompeuse — classes déséquilibrées |

`evaluate_loader` évalue le modèle sur l'ensemble d'un dataloader en
concaténant tous les batches avant le calcul des métriques.


In [ ]:
def compute_IoU_binary(preds, masks):
    """Intersection over Union pour la classe côte.
    Métrique standard en segmentation sémantique.
    IoU = TP / (TP + FP + FN)
    """
    pb    = (preds > 0.5).float()          # binarisation des probabilités
    inter = (pb * masks).sum()             # vrais positifs (TP)
    union = pb.sum() + masks.sum() - inter # TP + FP + FN
    return (inter / (union + 1e-8)).item() # +1e-8 pour éviter division par zéro

def compute_mIoU_binary(preds, masks):
    """mIoU binaire : moyenne de l'IoU des deux classes (fond + côte).
    Métrique standard dans la littérature de segmentation sémantique.
    """
    pb = (preds > 0.5).float()

    # IoU classe 1 : côte
    tp    = (pb * masks).sum()
    fp    = (pb * (1 - masks)).sum()
    fn    = ((1 - pb) * masks).sum()
    iou_coast = tp / (tp + fp + fn + 1e-8)

    # IoU classe 0 : fond
    tn    = ((1 - pb) * (1 - masks)).sum()
    iou_bg = tn / (tn + fn + fp + 1e-8)
    return ((iou_coast + iou_bg) / 2).item(), iou_coast.item(), iou_bg.item()


def compute_precision(preds, masks):
    """Précision = TP / (TP + FP)
    Parmi tous les pixels prédits comme côte, quelle fraction est correcte ?
    Une précision faible indique beaucoup de faux positifs (sur-détection).
    """
    pb = (preds > 0.5).float()
    tp = (pb * masks).sum()               # pixels correctement prédits côte
    fp = (pb * (1 - masks)).sum()         # pixels prédits côte mais qui sont fond
    return (tp / (tp + fp + 1e-8)).item()

def compute_recall(preds, masks):
    """Recall (sensibilité) = TP / (TP + FN)
    Parmi tous les pixels de côte réels, quelle fraction est détectée ?
    Un recall faible indique que le modèle manque une partie de la côte.
    """
    pb = (preds > 0.5).float()
    tp = (pb * masks).sum()               # pixels côte correctement détectés
    fn = ((1 - pb) * masks).sum()         # pixels côte manqués
    return (tp / (tp + fn + 1e-8)).item()

def compute_f1(preds, masks):
    """F1-Score = 2 * Précision * Recall / (Précision + Recall)
    Moyenne harmonique de la précision et du recall.
    Métrique de choix pour les classes déséquilibrées car elle pénalise
    les cas extrêmes (tout prédire côte ou tout prédire fond).
    """
    p, r = compute_precision(preds, masks), compute_recall(preds, masks)
    return 2 * p * r / (p + r + 1e-8)

def compute_accuracy(preds, masks):
    """Accuracy = (TP + TN) / total
    Fraction de pixels correctement classifiés.
    Métrique trompeuse sur les classes déséquilibrées : un modèle prédisant
    tout 'fond' obtient ~99% d'accuracy malgré une détection nulle de la côte.
    """
    return ((preds > 0.5).float() == masks).float().mean().item()


def evaluate_loader(model, dataloader, device, seuil=0.5):
    """Évalue le modèle sur l'ensemble d'un dataloader.
         Le seuil de décision =0.5 par défaut)
    """
    model.eval()  # désactive le dropout et le BatchNorm en mode entraînement
    probs_list, masks_list = [], []

    with torch.no_grad():  # désactive le calcul des gradients pour économiser mémoire et temps
        for imgs, masks in dataloader:
            # Inférence : logits → sigmoid → probabilités
            probs = torch.sigmoid(model(imgs.to(device)))
            probs_list.append(probs.cpu())                  # rapatrie sur CPU pour libérer la VRAM
            masks_list.append(masks.unsqueeze(1).cpu())     # unsqueeze : [B,H,W] → [B,1,H,W]

    # Concaténation de tous les batches en un seul tenseur
    all_probs = torch.cat(probs_list)   # [N, 1, H, W]
    all_masks = torch.cat(masks_list)   # [N, 1, H, W]
    # Binarisation au seuil choisi
    preds_bin = (all_probs > seuil).float()

    print(f'  {all_probs.shape[0]} images | GT={all_masks.sum():,.0f}px | '
          f'Pred={preds_bin.sum():,.0f}px')

    miou, iou_coast, iou_bg = compute_mIoU_binary(preds_bin, all_masks)

    m = {
        'IoU_côte'  : iou_coast,   # ce que vous appeliez mIoU
        'mIoU'      : miou,        # vrai mIoU (moyenne des deux classes)
        'IoU_fond'  : iou_bg,
        'precision' : compute_precision(preds_bin, all_masks),
        'recall'    : compute_recall(preds_bin, all_masks),
        'f1'        : compute_f1(preds_bin, all_masks),
        'accuracy'  : compute_accuracy(preds_bin, all_masks),
        'pixels_pred': preds_bin.sum().item(),
    }
    print(f'  IoU_côte={m["IoU_côte"]:.3f} | mIoU={m["mIoU"]:.3f} | '
      f'F1={m["f1"]:.3f} | Prec={m["precision"]:.3f} | Rec={m["recall"]:.3f}')
    return m

## 7 — Diagnostic pré-entraînement

Ce diagnostic vérifie que le `pos_weight` est bien calibré **avant** de lancer
l'entraînement complet (coûteux en temps sur CPU).

Un UNet non entraîné (poids aléatoires) prédit des probabilités quasi-uniformes
autour de 0.5. Le nombre de pixels prédits comme côte (>0.5) devrait être
comparable au nombre réel de pixels côte dans le batch :

- `pixels_prédits ≈ pixels_GT` (±facteur 2-5) → pos_weight bien calibré ✔
- `pixels_prédits >> pixels_GT` → pos_weight trop élevé, baisser
- `pixels_prédits ≈ 0` → pos_weight trop faible, augmenter


In [ ]:
# Instanciation d'un UNet non entraîné (poids aléatoires) pour vérifier
# que le pos_weight est bien calibré AVANT de lancer l'entraînement.
# Un modèle non entraîné prédit de façon quasi-uniforme (~0.5 partout),
# ce qui sert de référence pour ajuster pos_weight.
_model_check = UNet().to(device)
_model_check.eval()

for imgs, masks in train_dl:
    with torch.no_grad():
        probs = torch.sigmoid(_model_check(imgs.to(device)))
    pqurint(f'Proba min/max/mean : {probs.min():.3f} / {probs.max():.3f} / {probs.mean():.3f}')
    print(f'Pixels GT          : {masks.sum().item():,.0f}')
    # Objectif : pixels prédits ≈ pixels GT (facteur 2 à 5 acceptable)
    print(f'Pixels prédits >0.5: {(probs > 0.5).sum().item():,.0f}  (modèle non entraîné)')
    break

del _model_check  # libère la mémoire GPU

print(f'\nRésumé dataset filtré :')
print(f'  MIN_COAST_PIXELS = {MIN_COAST_PIXELS}')
print(f'  Train={len(train_dataset)} | Val={len(val_dataset)} | Test={len(test_dataset)}')
print(f'  pos_weight = {pos_weight_value:.1f}')

## 8 — Boucle d'entraînement

### Optimiseur Adam + CosineAnnealingLR
- **Adam** (`lr=1e-4`, `weight_decay=1e-5`) : optimiseur adaptatif standard
  pour la segmentation. Le weight_decay applique une régularisation L2
  pour limiter le surapprentissage.
- **CosineAnnealingLR** : fait décroître le learning rate de 1e-4 à 1e-6
  suivant une courbe cosinus sur toute la durée de l'entraînement.
  Les premières epochs explorent l'espace des poids avec un lr élevé,
  les dernières affinent avec un lr très faible.

### Gradient clipping
`clip_grad_norm_(1.0)` plafonne la norme des gradients à 1.0 pour éviter
les explosions de gradient qui feraient diverger l'entraînement.

### Stratégie de sauvegarde
Seul le modèle avec le **meilleur F1 sur la validation** est sauvegardé
(`best_model_v12.pth`). Ce n'est pas nécessairement le modèle de la dernière
epoch — utiliser le dernier modèle risquerait de suroptimiser sur le train set.

### Calcul de la loss de validation
La loss de validation est calculée tous les 5 epochs pour détecter un éventuel
surapprentissage : si la train loss continue de baisser pendant que la val loss
remonte, le modèle overfitte.


In [ ]:
#--------------------------------------------------------
# PARAMÈTRES À AJUSTER
num_epochs = 50
#--------------------------------------------------------

# Réinitialisation du modèle à chaque run pour repartir de zéro
model     = UNet().to(device)
# Tenseur pos_weight sur le même device que le modèle
pw        = torch.tensor([pos_weight_value]).to(device)
# Optimiseur Adam : adaptatif, bien adapté aux problèmes de segmentation.
# lr=1e-4 : taux d'apprentissage initial.
# weight_decay=1e-5 : régularisation L2 pour limiter le surapprentissage.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
# Scheduler cosinus : fait décroître le lr de 1e-4 à 1e-6 progressivement.
# Les dernières epochs affinent les poids avec un lr très faible.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

best_val_f1, loss_list, val_loss_list = -1.0, [], []
print(f'pos_weight={pos_weight_value:.1f} | epochs={num_epochs} | device={device}')

for epoch in range(num_epochs):
    model.train()  # mode entraînement : active BatchNorm et dropout
    running_loss, n_batches = 0.0, 0
    pbar = tqdm(train_dl, desc=f'Epoch {epoch+1}/{num_epochs}')

    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)  # transfert GPU
        optimizer.zero_grad()                             # remise à zéro des gradients accumulés
        loss = combined_loss(model(imgs), masks, pw)      # forward + calcul de la loss
        loss.backward()                                   # rétropropagation : calcul des gradients
        # Clipping des gradients : évite les explosions de gradient
        # (gradient très grand qui ferait diverger l'entraînement).
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()                                  # mise à jour des poids
        running_loss += loss.item()
        n_batches    += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.1e}'})

    scheduler.step()  # mise à jour du lr après chaque epoch
    avg = running_loss / n_batches
    loss_list.append(avg)  # historique pour tracer la courbe de loss
    print(f'Epoch {epoch+1:3d}/{num_epochs} | Loss={avg:.4f} | lr={scheduler.get_last_lr()[0]:.2e}')

    # Validation tous les 5 epochs (et à la dernière epoch).
    # Évaluation sur val_dl uniquement (pas test_dl) pour éviter
    # un biais de sélection du modèle sur le jeu de test.
    if (epoch + 1) % 5 == 0 or (epoch + 1) == num_epochs:
        print(f'  Validation...')
        
        # --- Calcul de la loss de validation ---
        model.eval()
        val_running_loss, val_n_batches = 0.0, 0
        with torch.no_grad():
            for imgs_v, masks_v in val_dl:
                imgs_v, masks_v = imgs_v.to(device), masks_v.to(device)
                val_loss = combined_loss(model(imgs_v), masks_v, pw)
                val_running_loss += val_loss.item()
                val_n_batches    += 1
        avg_val_loss = val_running_loss / val_n_batches
        val_loss_list.append((epoch + 1, avg_val_loss))
        print(f'  Val Loss={avg_val_loss:.4f}')
        # ----------------------------------------
        
        val_m = evaluate_loader(model, val_dl, device)
        # Sauvegarde du meilleur modèle selon le F1 de validation
        if val_m['f1'] > best_val_f1:
            best_val_f1 = val_m['f1']
            torch.save(model.state_dict(), 'best_model_v12.pth')
            print(f'  Meilleur F1 val={best_val_f1:.3f} → sauvegardé')
# Sauvegarde du modèle final (dernière epoch, pas nécessairement le meilleur)
torch.save(model.state_dict(),
    r'C:\Users\Rhm\Documents\M2 SIC\Projet de recherche\dataset\test 3\modele\unet_modelv12_50.pth')
print('Entraînement terminé.')

## 9 — Courbes ROC et Precision-Recall

### Pourquoi deux courbes ?
- La **courbe ROC** (AUC) mesure la capacité globale du modèle à discriminer
  côte et fond, indépendamment du seuil. Sur un dataset très déséquilibré,
  l'AUC-ROC peut être artificiellement élevée car elle intègre les vrais négatifs
  (fond bien classifié, très nombreux).
- La **courbe Precision-Recall** est plus informative pour les classes déséquilibrées
  car elle ne prend pas en compte les vrais négatifs. C'est elle qui est utilisée
  pour **déterminer le seuil optimal**.

### Seuil optimal
Le seuil optimal est la valeur de probabilité qui **maximise le F1-score** sur
la courbe Precision-Recall :
```
F1 = 2 × Precision × Recall / (Precision + Recall)
seuil_optimal = argmax(F1)
```
Ce seuil est utilisé pour l'évaluation finale (section 11).
**Note :** ce seuil (0.935) est calculé sur les tuiles de validation.
En inférence sur image complète (sliding window), un seuil plus bas (0.5)
est nécessaire car les zones de jonction entre tuiles produisent des probabilités
plus modérées, ce qui fragmente la ligne à seuil élevé.


In [ ]:


def plot_roc(model, dataloader, device, save_path='roc_curve_v12_'+str(num_epochs)+'.png'):
    """Trace les courbes ROC et Precision-Recall et retourne le seuil
    qui maximise le F1-Score.

    La courbe ROC (Receiver Operating Characteristic) trace le taux de vrais
    positifs (recall) en fonction du taux de faux positifs à chaque seuil.
    L'AUC (aire sous la courbe) mesure la capacité globale du modèle à
    discriminer côte et fond, indépendamment du seuil choisi.

    La courbe Precision-Recall est plus adaptée aux classes déséquilibrées
    car elle ne prend pas en compte les vrais négatifs (fond, majoritaire).
    C'est elle qui est utilisée pour déterminer le seuil optimal.
    """
    model.eval()
    all_probs_list, all_labels_list = [], []

    with torch.no_grad():
        for imgs, masks in dataloader:
            # Récupère les probabilités et les labels à plat (1D)
            probs  = torch.sigmoid(model(imgs.to(device))).cpu().numpy().flatten()
            labels = masks.numpy().flatten()
            all_probs_list.append(probs)
            all_labels_list.append(labels)

    all_probs  = np.concatenate(all_probs_list)   # vecteur de toutes les probabilités prédites
    all_labels = np.concatenate(all_labels_list)  # vecteur de toutes les vérités terrain

    # Courbe Precision-Recall : calcule précision et recall à chaque seuil possible
    precisions, recalls, thresholds = precision_recall_curve(all_labels, all_probs)
    # [:-1] : precision_recall_curve ajoute un point (prec=1, rec=0) sans seuil associé,
    # on l'exclut pour aligner les indices avec thresholds.
    f1_scores     = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    optimal_idx   = np.argmax(f1_scores)       # indice du seuil maximisant le F1
    optimal_seuil = thresholds[optimal_idx]    # valeur du seuil optimal

    # Courbe ROC : uniquement pour calculer l'AUC (indicateur global)
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # Graphe 1 : courbe ROC
    ax1.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_auc:.3f}')
    ax1.plot([0, 1], [0, 1], 'k--', lw=1)  # ligne de référence (modèle aléatoire)
    ax1.set_xlabel('FPR (taux faux positifs)'); ax1.set_ylabel('TPR (recall)')
    ax1.set_title('Courbe ROC')
    ax1.legend(); ax1.grid(alpha=0.3)

    # Graphe 2 : courbe Precision-Recall avec seuil optimal
    ax2.plot(recalls, precisions, color='coral', lw=2)
    ax2.scatter(recalls[optimal_idx], precisions[optimal_idx],
                color='red', zorder=5,
                label=f'Seuil de binarisation = {optimal_seuil:.3f}\n'
                    f'F1={f1_scores[optimal_idx]:.3f} '
                    f'Prec={precisions[optimal_idx]:.3f} '
                    f'Rec={recalls[optimal_idx]:.3f}')
    ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
    ax2.set_title('Courbe Precision-Recall')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()

    print(f'AUC             : {roc_auc:.3f}')
    print(f'Seuil optimal F1: {optimal_seuil:.3f}')
    print(f'  Precision     : {precisions[optimal_idx]:.3f}')
    print(f'  Recall        : {recalls[optimal_idx]:.3f}')
    print(f'  F1            : {f1_scores[optimal_idx]:.3f}')
    return optimal_seuil

seuil_optimal = plot_roc(model, val_dl, device)

## 10 — Courbe de loss (train + validation)

Cette figure permet de diagnostiquer visuellement le comportement de l'entraînement :

- **Train loss et val loss proches et décroissantes** → entraînement sain, pas d'overfitting ✔
- **Val loss remonte pendant que train loss descend** → surapprentissage
- **Les deux loss stagnent tôt** → learning rate trop faible ou capacité du modèle insuffisante

La val loss est calculée tous les 5 epochs (checkpoints) et non à chaque epoch
pour limiter le temps de calcul sur CPU.


In [ ]:
# Tracé de la loss moyenne par epoch.
# Une loss qui décroît régulièrement indique une bonne convergence.
# Un plateau précoce peut signaler un lr trop faible ou un pos_weight inadapté.
val_epochs = [x[0] for x in val_loss_list]
val_losses = [x[1] for x in val_loss_list]

plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), loss_list, 
         marker='o', markersize=3, label='Train loss')
plt.plot(val_epochs, val_losses, 
         marker='s', markersize=5, linestyle='--', color='red', label='Val loss')
plt.title('Loss moyenne (BCE + Dice) — UNet V12')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('loss_unet_v12_transfo_semseg' + str(num_epochs) + '.png', dpi=150)
plt.show()

## 11 — Évaluation finale

Le **meilleur modèle** sauvegardé pendant l'entraînement (meilleur F1 val)
est chargé et évalué sur les trois splits avec le seuil optimal calculé
à la section 9.

**Interprétation des résultats :**
- Un écart faible entre train, val et test confirme l'absence de surapprentissage
  et la bonne généralisation du modèle.
- Un test légèrement meilleur que le train est possible avec de petits splits
  et indique simplement une variabilité naturelle entre les ensembles.
- L'accuracy (~0.999) est trompeuse sur ce dataset : avec <1% de pixels côte,
  prédire tout 'fond' donnerait déjà ~0.994. Se concentrer sur F1 et IoU côte.


In [ ]:
# Chargement du meilleur modèle sauvegardé pendant l'entraînement
# (celui avec le meilleur F1 sur val, pas nécessairement le dernier epoch)
model.load_state_dict(torch.load('best_model_v12.pth', map_location=device))
model.eval()

print('='*70)
print('ÉVALUATION FINALE')
print('='*70)

# Évaluation sur les trois splits avec le seuil optimal déterminé par la ROC
all_metrics = {}
for name, loader in [('TRAIN', train_dl), ('VALIDATION', val_dl), ('TEST', test_dl)]:
    print(f'\n{name}')
    all_metrics[name] = evaluate_loader(model, loader, device, seuil=seuil_optimal)

# Tableau de synthèse des métriques
print('\n' + '='*60)
print(f'Meilleur F1 Validation : {best_val_f1:.3f}')
print('Dataset      | IoU côte | mIoU  | Prec  | Rec   | F1    | Acc')
print('-'*60)
for name, m in all_metrics.items():
    print(f'{name:12} | {m["IoU_côte"]:5.3f} | {m["mIoU"]:5.3f} | {m["precision"]:5.3f} | '
          f'{m["recall"]:5.3f} | {m["f1"]:5.3f} | {m["accuracy"]:5.3f}')

## 12 — Inférence sur nouvelles images

Cette section applique le modèle entraîné à une image satellite complète
(4695×2344 pixels) qui n'a pas été vue pendant l'entraînement.

### Stratégie sliding window avec stride=128
L'image est découpée en tuiles 256×256 avec un chevauchement de 50% (stride=128).
Pour chaque pixel couvert par plusieurs tuiles, les probabilités prédites sont
**moyennées** avant binarisation. Cela lisse les artefacts aux jonctions
entre tuiles.

### Paramètres d'inférence
| Paramètre | Valeur | Justification |
|---|---|---|
| `STRIDE` | 128 | 50% de chevauchement — bon compromis qualité/vitesse |
| `SEUIL` | 0.5 | Seuil conservateur pour maintenir la continuité de la ligne |
| `MIN_COMPONENT_SIZE` | 5000 | Supprime les fragments courts, conserve la ligne principale |

### Post-traitement : filtrage des composantes connexes
Après binarisation, des faux positifs isolés (ombres, structures urbaines)
subsistent. Le filtrage par composantes connexes (`scipy.ndimage`) supprime
tous les fragments de moins de `MIN_COMPONENT_SIZE` pixels, ne conservant
que la ligne de côte principale qui est une structure continue.

### Sortie
Deux fichiers GeoTIFF géoréférencés (EPSG:2154 Lambert 93) sont produits :
- `_mosaic_...tif` : masque binaire (0=fond, 255=côte) — pour les analyses
- `_probmap_...tif` : carte de probabilités float32 — pour le diagnostic


In [ ]:

# --- Chemins d'entrée/sortie pour l'inférence ---
input_folder       = r'C:\Users\Rhm\Documents\M2 SIC\Projet de recherche\dataset\test 3\dataset_stride\Input_pred'
mask_output_root   = r'C:\Users\Rhm\Documents\M2 SIC\Projet de recherche\dataset\test 3\dataset_stride\Masques_Georef_V12'
mosaic_output_root = r'C:\Users\Rhm\Documents\M2 SIC\Projet de recherche\dataset\test 3\dataset_stride\Masks_pred'
tile_output_root   = r'C:\Users\Rhm\Documents\M2 SIC\Projet de recherche\dataset\test 3\dataset_stride\Tuiles_Georef'

tile_size   = 256   # taille des tuiles (doit correspondre à l'entraînement)
STRIDE      = 128   # chevauchement 50% — réduire à 64 pour encore plus de lissage (plus lent)
SEUIL       = 0.935 #seuil_optimal

for d in [mask_output_root, mosaic_output_root, tile_output_root]:
    os.makedirs(d, exist_ok=True)

infer_transform = get_val_transform()

#model.load_state_dict(torch.load(
#    r'C:\Users\Rhm\Documents\M2 SIC\Projet de recherche\dataset\test 3\modele\unet_modelv12_50.pth',
#    map_location=device))
model.load_state_dict(torch.load(
    'best_model_v12.pth',
    map_location=device))
model.to(device).eval()

for tif_file in [f for f in os.listdir(input_folder) if f.lower().endswith('.tif')]:
    input_path = os.path.join(input_folder, tif_file)
    base_name  = os.path.splitext(tif_file)[0] + str(num_epochs)

    with rasterio.open(input_path) as src:
        W, H           = src.width, src.height
        profile        = src.profile
        crs            = src.crs if src.crs else CRS.from_string('EPSG:2154')
        transform_glob = src.transform
        print(f'\n{tif_file}  ({W}x{H}) — stride={STRIDE}px')

        # --- Tableaux d'accumulation pour le moyennage ---
        # accum : somme des probabilités prédites sur chaque pixel
        # count : nombre de tuiles ayant couvert chaque pixel
        accum = np.zeros((H, W), dtype=np.float32)
        count = np.zeros((H, W), dtype=np.float32)

        # --- Boucle de découpe avec stride ---
        tops  = list(range(0, H - tile_size + 1, STRIDE)) + ([H - tile_size] if H % STRIDE != 0 else [])
        lefts = list(range(0, W - tile_size + 1, STRIDE)) + ([W - tile_size] if W % STRIDE != 0 else [])
        # La dernière tuile est décalée pour couvrir exactement le bord,
        # évitant ainsi les tuiles tronquées qui nécessiteraient un padding spécial.

        total_tiles = len(tops) * len(lefts)
        print(f'  Nombre de tuiles à traiter : {total_tiles}')

        for top in tops:
            for left in lefts:
                # Lecture de la tuile (toujours tile_size × tile_size grâce au calcul des bords)
                tile_data = src.read(window=Window(left, top, tile_size, tile_size))

                # Prédiction
                img_hwc   = np.moveaxis(tile_data[:3].astype(np.uint8), 0, -1)
                augmented = infer_transform(image=img_hwc)
                tensor    = augmented['image'].unsqueeze(0).to(device)

                with torch.no_grad():
                    prob = torch.sigmoid(model(tensor))[0, 0].cpu().numpy()  # [tile_size, tile_size]

                # Accumulation dans la zone correspondante de l'image complète
                # top:top+tile_size et left:left+tile_size sont toujours dans les bornes
                # car tops et lefts sont calculés pour rester dans [0, H-tile_size] et [0, W-tile_size]
                accum[top:top+tile_size, left:left+tile_size] += prob
                count[top:top+tile_size, left:left+tile_size] += 1.0

        # --- Moyennage et binarisation finale ---
        # On divise par le nombre de tuiles ayant couvert chaque pixel.
        # Les pixels centraux sont couverts par plusieurs tuiles → meilleure estimation.
        # Les pixels de bord sont couverts par moins de tuiles mais count >= 1 partout.
        mean_prob  = accum / np.maximum(count, 1.0)
        final_mask = (mean_prob > SEUIL).astype(np.uint8)
        # --- DIAGNOSTIC : à lancer UNE FOIS pour trouver le bon seuil de filtrage---
        #labeled, num_features = ndimage.label(final_mask)
        #component_sizes = np.array(ndimage.sum(
        #    final_mask, labeled, range(1, num_features + 1)))
        #print(f'  Nombre de composantes      : {num_features}')
        #print(f'  5 plus grandes composantes : {sorted(component_sizes, reverse=True)[:5]}')
        #print(f'  Taille médiane             : {np.median(component_sizes):.0f}')
        #print(f'  Taille max                 : {component_sizes.max():.0f}')
        # --- FIN DIAGNOSTIC ---

        # --- Filtrage des composantes connexes --- 
        # Supprimer tous les fragments sauf la ligne principale
        MIN_COMPONENT_SIZE = #5000 avec seuil 0.5  # 175 avec seuil optimal

        labeled, num_features = ndimage.label(final_mask)
        component_sizes = ndimage.sum(
            final_mask, labeled, range(1, num_features + 1))

        final_mask = np.zeros_like(final_mask)
        for i, size in enumerate(component_sizes, start=1):
            if size >= MIN_COMPONENT_SIZE:
                final_mask[labeled == i] = 1

        print(f'  Composantes trouvées   : {num_features}')
        print(f'  Composantes conservées : {(np.array(component_sizes) >= MIN_COMPONENT_SIZE).sum()}')
        print(f'  Pixels côte finaux     : {final_mask.sum():,}')
        # ---
        print(f'  Pixels côte détectés : {final_mask.sum():,} / {H*W:,} '
            f'({100*final_mask.sum()/(H*W):.3f}%)')
        
        
        # ---Régkage épaisseur ligne de côtes pour une meilleure visualisation --- 
        
        DILATION_RADIUS = 5  # épaisseur en pixels de chaque côté
                            # 2 → ligne de 5px, 3 → ligne de 7px
        # Création d'un élément structurant circulaire
        struct = np.ones((2*DILATION_RADIUS+1, 2*DILATION_RADIUS+1), dtype=bool)

        # Dilatation du masque binaire
        final_mask_display = binary_dilation(
            final_mask.astype(bool), structure=struct).astype(np.uint8)

        # Sauvegarde de la version épaissie (pour le rapport)
        display_path = os.path.join(
            mosaic_output_root,
            f'{base_name}_display_radius{DILATION_RADIUS}.tif')
        out_profile_display = {
            'driver': 'GTiff', 'dtype': 'uint8',
            'width': W, 'height': H, 'count': 1,
            'crs': crs, 'transform': transform_glob,
        }   
        with rasterio.open(display_path, 'w', **out_profile_display) as dst:
            dst.write((final_mask_display * 255)[np.newaxis, :, :])
        print(f'  Version rapport (radius={DILATION_RADIUS}px) : {display_path}')

        # --- Sauvegarde de la mosaïque géoréférencée ---
        mosaic_path = os.path.join(mosaic_output_root,
                                f'{base_name}_mosaic_transfo_v12_stride{STRIDE}.tif')
        out_profile = {
            'driver'   : 'GTiff',
            'dtype'    : 'uint8',
            'width'    : W,
            'height'   : H,
            'count'    : 1,
            'crs'      : crs,
            'transform': transform_glob,
        }
        with rasterio.open(mosaic_path, 'w', **out_profile) as dst:
            dst.write((final_mask * 255)[np.newaxis, :, :])
        #with rasterio.open(mosaic_path, 'w', **out_profile) as dst:
        #    dst.write(final_mask[np.newaxis, :, :])
        print(f'  Mosaïque : {mosaic_path}')

        # --- Sauvegarde optionnelle de la carte de probabilités brutes ---
        # Utile pour visualiser les zones d'incertitude du modèle
        prob_path = os.path.join(mosaic_output_root,
                                 f'{base_name}_probmap_v12_stride{STRIDE}.tif')
        prob_profile = out_profile.copy()
        prob_profile['dtype'] = 'float32'
        with rasterio.open(prob_path, 'w', **prob_profile) as dst:
            dst.write(mean_prob[np.newaxis, :, :])
        print(f'  Carte proba : {prob_path}')

print('\nInférence terminée.')

In [ ]:


mosaic_path = r'C:\Users\Rhm\Documents\M2 SIC\Projet de recherche\dataset\test 3\dataset_stride\Masks_pred\decoupe50_mosaic_transfo_v12_stride128.tif'

with rasterio.open(mosaic_path) as src:
    data = src.read(1)
    print(f'dtype    : {data.dtype}')
    print(f'min/max  : {data.min()} / {data.max()}')
    print(f'pixels>0 : {(data > 0).sum():,}')

## Annexe — Visualisation qualitative des prédictions

Cette cellule affiche côte à côte pour une tuile du train set :
1. L'image satellite originale (dénormalisée pour l'affichage)
2. Le masque de vérité terrain (ground truth)
3. La carte de probabilités brutes (heatmap 'hot' : rouge=fort, noir=faible)
4. Le masque binarisé au seuil 0.5
5. L'histogramme des probabilités — idéalement bimodal (pics vers 0 et 1)
   indiquant un modèle confiant dans ses prédictions

Cette visualisation permet une inspection rapide de la qualité des prédictions
sans attendre l'évaluation complète sur le jeu de test.


In [ ]:
# Visualisation qualitative sur une tuile du train set.
# Permet d'inspecter visuellement la qualité des prédictions.
model.eval()
plt.figure(figsize=(20, 4))

for imgs, masks in train_dl:
    if masks.sum() > 50:  # on cherche une tuile contenant de la côte
        imgs, masks = imgs.to(device), masks.to(device)
        with torch.no_grad():
            probs = torch.sigmoid(model(imgs))  # probabilités par pixel

        # Dénormalisation pour affichage : annule la normalisation ImageNet
        img_vis = imgs[0].cpu().permute(1, 2, 0).numpy()  # [3,H,W] → [H,W,3]
        img_vis = np.clip(img_vis * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)

        mask_vis = masks[0].cpu() if masks.dim() == 3 else masks[0, 0].cpu()
        pred_vis = probs[0, 0].cpu()  # carte de probabilités [H,W]

        # Colonne 1 : image satellite originale
        plt.subplot(1, 5, 1); plt.imshow(img_vis); plt.title('Image'); plt.axis('off')
        # Colonne 2 : masque de vérité terrain (GT)
        plt.subplot(1, 5, 2); plt.imshow(mask_vis, cmap='gray'); plt.title(f'GT {mask_vis.sum():.0f}px'); plt.axis('off')
        # Colonne 3 : carte de probabilités brutes (heatmap)
        plt.subplot(1, 5, 3); plt.imshow(pred_vis, cmap='hot'); plt.title(f'Proba {pred_vis.min():.2f}-{pred_vis.max():.2f}'); plt.axis('off')
        # Colonne 4 : masque binarisé au seuil 0.5
        plt.subplot(1, 5, 4); plt.imshow((pred_vis > 0.5).float(), cmap='gray'); plt.title('Binarisé 0.5'); plt.axis('off')
        # Colonne 5 : histogramme des probabilités (idéalement bimodal : pics vers 0 et 1)
        plt.subplot(1, 5, 5); plt.hist(pred_vis.flatten().numpy(), bins=50); plt.title('Histogramme')
        plt.tight_layout()
        plt.savefig('visu_prediction_v12_' + str(num_epochs) + '.png', dpi=150)
        plt.show()
        break